In [ ]:
import requests

# 1. Definir la URL de Open-Meteo exacta para Garagoa (Valle de Tenza)
# Coordenadas de Garagoa: Latitud 5.08, Longitud -73.36
# Pedimos: temperatura actual, lluvia (precipitación) y humedad relativa
url = "https://api.open-meteo.com/v1/forecast?latitude=5.08&longitude=-73.36&current=temperature_2m,precipitation,relative_humidity_2m&timezone=America%2FBogota"

# 2. Hacer la petición a la web (Consumo de API / Scraping)
print("Conectando con Open-Meteo para extraer datos de Garagoa...")
respuesta = requests.get(url)
datos_crudos = respuesta.json() # Convertimos la respuesta de la web a un formato manejable

# 3. ESTRUCTURA DE DATOS (Requisito de la asignatura): TABLA HASH
# En Python, una Tabla Hash se implementa usando un Diccionario.
# Usamos "Garagoa_Nodo_1" como clave (Key) para tener un tiempo de búsqueda O(1).
tabla_hash_clima = {
    "Garagoa_Nodo_1": {
        "temperatura_C": datos_crudos['current']['temperature_2m'],
        "lluvia_mm": datos_crudos['current']['precipitation'],
        "humedad_relativa_%": datos_crudos['current']['relative_humidity_2m'],
        "ultima_actualizacion": datos_crudos['current']['time']
    }
}

# 4. Imprimir la Tabla Hash (Evidencia para el Dashboard)
print("\n=== DASHBOARD CONTEXTUAL: PREVENCIÓN DE DESLIZAMIENTOS ===")
print("Búsqueda rápida en Tabla Hash O(1) arrojó los siguientes resultados:")
print(tabla_hash_clima["Garagoa_Nodo_1"])

# 5. Lógica de validación cruzada (Requisito del proyecto Integrador)
lluvia_actual = tabla_hash_clima["Garagoa_Nodo_1"]["lluvia_mm"]
humedad_actual = tabla_hash_clima["Garagoa_Nodo_1"]["humedad_relativa_%"]

if lluvia_actual > 0 or humedad_actual > 85:
    print("\n⚠️ ALERTA MACRO-CLIMÁTICA: Lluvia o humedad altísima detectada en la zona.")
    print("El suelo podría estar saturándose de agua. Riesgo de reducción de fricción en la ladera.")
else:
    print("\n✅ Clima externo seco. La saturación de agua por lluvia es nula en este instante.")

Conectando con Open-Meteo para extraer datos de Garagoa...

=== DASHBOARD CONTEXTUAL: PREVENCIÓN DE DESLIZAMIENTOS ===
Búsqueda rápida en Tabla Hash O(1) arrojó los siguientes resultados:
{'temperatura_C': 22.7, 'lluvia_mm': 0.1, 'humedad_relativa_%': 60, 'ultima_actualizacion': '2026-11-06T14:30'}

⚠️ ALERTA MACRO-CLIMÁTICA: Lluvia o humedad altísima detectada en la zona.
El suelo podría estar saturándose de agua. Riesgo de reducción de fricción en la ladera.


In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime

# URL de Open-Meteo para Garagoa pidiendo clima actual + lluvia por hora del último día
url = "https://api.open-meteo.com/v1/forecast?latitude=5.08&longitude=-73.36&current=temperature_2m,precipitation,relative_humidity_2m&hourly=precipitation&past_days=1&forecast_days=1&timezone=America%2FBogota"

# Lista donde acumularemos los registros a lo largo del tiempo
historial_registros = []

# Simulamos 3 consultas periódicas (en campo real esto correría cada 15 o 30 minutos)
for intento in range(3):
    print(f"Realizando extracción #{intento + 1}...")
    respuesta = requests.get(url)
    datos = respuesta.json()

    # Calculamos la lluvia acumulada de las últimas 24 horas sumando la lista horaria
    lluvias_horarias = datos['hourly']['precipitation'][:24]
    lluvia_acumulada_24h = round(sum(lluvias_horarias), 2)

    # ESTRUCTURA DE DATOS: Tabla Hash (Diccionario) para acceso O(1)
    tabla_hash_clima = {
        "Garagoa_Nodo_1": {
            "fecha_hora": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "temperatura_C": datos['current']['temperature_2m'],
            "humedad_relativa_%": datos['current']['relative_humidity_2m'],
            "lluvia_instantanea_mm": datos['current']['precipitation'],
            "lluvia_acumulada_24h_mm": lluvia_acumulada_24h
        }
    }

    # Guardamos el registro extraído de la Tabla Hash en nuestro historial
    registro_actual = tabla_hash_clima["Garagoa_Nodo_1"]
    historial_registros.append(registro_actual)

    # Respetamos los límites de tasa (Rate Limits - Unidad 3) esperando 2 segundos entre peticiones
    time.sleep(2)

# Convertimos el historial acumulado a una tabla de Pandas y lo exportamos a CSV
df_clima = pd.DataFrame(historial_registros)
df_clima.to_csv("historico_clima_garagoa.csv", index=False)

print("\n=== DATASET CLIMÁTICO ACUMULADO GUARDADO ===")
print(df_clima)

Realizando extracción #1...
Realizando extracción #2...
Realizando extracción #3...

=== DATASET CLIMÁTICO ACUMULADO GUARDADO ===
            fecha_hora  temperatura_C  humedad_relativa_%  \
0  2026-11-18 01:56:07           15.9                  95   
1  2026-11-18 01:56:09           15.9                  95   
2  2026-11-18 01:56:11           15.9                  95   

   lluvia_instantanea_mm  lluvia_acumulada_24h_mm  
0                    0.0                      1.5  
1                    0.0                      1.5  
2                    0.0                      1.5  
